> **日本語版について**: このノートブックは、[jpmorganchase/python-training](https://github.com/jpmorganchase/python-training) の日本語版です（Apache License 2.0）。原文の翻訳を基礎に、日本語版独自の補足・追加・修正を含む場合があり、独自の部分にはその旨を明記しています。
>
> この章は翻訳のみで、コードは原文から変更していません。

# 例2：ファイルを扱う

まず、データファイルがあるか確認します。`os` ライブラリを使います。

In [ ]:
from os import listdir
from os.path import join

listdir('.')

In [ ]:
listdir('../data')

`pandas` の `read_csv` 関数を使います。

In [ ]:
import pandas as pd
pd.read_csv?

`pandas` には、他にどんな「read」系の関数があるか見てみます。

In [ ]:
import re
regex = re.compile(r'read')
list(filter(regex.match, dir(pd)))

ここでは `read_csv` だけを使います。データは [openflights.org](openflights.org) のものを調べます。

In [ ]:
routes = pd.read_csv(join('..', 'data', 'routes.csv'))
airports = pd.read_csv(join('..', 'data', 'airports.csv'))

In [ ]:
routes.head()

In [ ]:
routes.info()

欠損している箇所がありそうです。図にして確認します。

In [ ]:
import missingno as msno

`missingno` は、データの欠損を図で確認するためのライブラリです。ここでは一部の機能だけを使います。

In [ ]:
%matplotlib inline
msno.matrix(routes)

これで欠損の様子が一目で分かります。右端の線は、データがどれだけ揃っているかを示します。次に `airports` も見てみます。

In [ ]:
airports.info()

In [ ]:
msno.matrix(airports)

全体にきれいなデータです。欠損も少なめです。路線は約68,000件、空港は世界に約7,000件あります。本物のデータか確かめるために、ラガーディア空港（LGA）を探してみます。

In [ ]:
airports[airports['IATA'] == 'LGA']

見つかりました。次に3つ探します。世界で一番標高が高い空港、国名がアルファベット順で最後の国で一番高い空港、オーストラリアで一番高い空港です。

In [ ]:
highest = airports.sort_values('altitude', ascending=False).iloc[0]
highestInLastAlpha = airports.sort_values(['country', 'altitude'], ascending=[False, False]).iloc[0]
highestInAustralia = airports[airports['country'] == 'Australia'].sort_values('altitude', ascending=False).iloc[0]

In [ ]:
print(f'Highest airport: \n{highest}\n\n')
print(f'Highest airport in the country that comes last alphabetically: \n{highestInLastAlpha}\n\n')
print(f'Highest airport in Australia: \n{highestInAustralia}\n\n')

次は実用的な集計をします。登録されている路線データを出発地と目的地の組み合わせごとに数え、件数が多い上位10組を調べます。

In [ ]:
busiest = routes.groupby(['source', 'dest']).count()['airline_id'].nlargest(10)
busiest

空港名を付け足します。

In [ ]:
busiest = pd.DataFrame(busiest).reset_index()
busiest.head()

In [ ]:
for sd in ['source', 'dest']:
    busiest = busiest.merge(airports[['IATA', 'name']].set_index('IATA'), how='left', left_on=sd, right_on='IATA')

busiest